# fact_search_hit_poc (factless event)

Grain: one video returned for one keyword in one search run (`fetched_at`, `keyword_normalized`, `video_id`). `hit_count = 1` (additive), `result_rank` non-additive.

The live `fact_search_hit` dedups on the **keys** (`keyword_key`, `video_key`); this one dedups on the **natural** keys, so two different videos that are both still −1 stay two rows.

**Incremental:** results fetched in the window (Vietnam day of `fetched_at`) **+** rows whose `keyword_key` / `video_key` / `channel_key` is still −1.

Task `fact_search_hit_poc` of the job "2. gold (poc tables)", after the dims it reads. Keys come from the `*_scd1` dims (same `xxhash64` keys as the live dims); no match → −1. Test table: the live fact and the agg tables are unchanged.

In [ ]:
%run ./_udfs

In [ ]:
%run ./_scd

# Set up environment

In [ ]:
for layer in ("silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
dbutils.widgets.dropdown("load_type", "incremental", ["incremental", "full"])
dbutils.widgets.text("from_date", "")   # yyyy-MM-dd, Vietnam day, inclusive; empty = yesterday
dbutils.widgets.text("to_date", "")     # exclusive; empty = tomorrow

silver = dbutils.widgets.get("silver")
gold = dbutils.widgets.get("gold")
load_type = dbutils.widgets.get("load_type")
from_date, to_date = get_incremental_dates(dbutils.widgets.get("from_date"), dbutils.widgets.get("to_date"))

# First load of the table = full (poc update_load_type: full once, then incremental)
if not spark.catalog.tableExists(f"{gold}.fact_search_hit_poc"):
    load_type = "full"
print(f"load_type = {load_type}, window = [{from_date}, {to_date})")

# Read Source Table

In [ ]:
path = f"{gold}.fact_search_hit_poc"
search_results = (spark.table(f"{silver}.stg_search_results")
                  .filter("video_id IS NOT NULL")
                  .withColumn("keyword_normalized", F.expr("normalize_keyword(search_keyword)")))
dim_keyword = spark.table(f"{gold}.dim_keyword_scd1").filter("keyword_key <> -1")
dim_video = spark.table(f"{gold}.dim_video_scd1").filter("video_key <> -1")
dim_channel = spark.table(f"{gold}.dim_channel_scd1").filter("channel_key <> -1")

## Incremental: results fetched in the window + rows with an unresolved key

In [ ]:
if load_type == "incremental":
    fetched_day = F.to_date(F.from_utc_timestamp("fetched_at", TZ))
    in_window = search_results.filter((fetched_day >= from_date) & (fetched_day < to_date))
    if spark.catalog.tableExists(path):
        unresolved = (spark.table(path)
                      .filter("keyword_key = -1 OR video_key = -1 OR channel_key = -1")
                      .select("fetched_at", "keyword_normalized", "video_id"))
        in_window = in_window.unionByName(
            search_results.join(unresolved, ["fetched_at", "keyword_normalized", "video_id"], "left_semi"))
    search_results = in_window

# Transformation

## One row per grain

The same video can come back twice in one run (e.g. two result pages): keep the first (`id`).

In [ ]:
from pyspark.sql.window import Window

first_result = Window.partitionBy("fetched_at", "keyword_normalized", "video_id").orderBy("id")
hits = search_results.withColumn("rn", F.row_number().over(first_result)).filter("rn = 1")

## Join dims and select columns

In [ ]:
result_df = (
    hits.alias("hit")
    .join(dim_keyword.alias("keyword"), on=F.col("hit.keyword_normalized") == F.col("keyword.keyword_normalized"), how="left")
    .join(dim_video.alias("video"), on=F.col("hit.video_id") == F.col("video.video_id"), how="left")
    .join(dim_channel.alias("channel"), on=F.col("hit.channel_id") == F.col("channel.channel_id"), how="left")
    .select(
        F.col("hit.fetched_at").cast("timestamp").alias("fetched_at"),
        F.col("hit.keyword_normalized").cast("string").alias("keyword_normalized"),
        F.col("hit.video_id").cast("string").alias("video_id"),
        F.coalesce(F.col("keyword.keyword_key"), F.lit(-1)).cast("bigint").alias("keyword_key"),
        F.coalesce(F.col("video.video_key"), F.lit(-1)).cast("bigint").alias("video_key"),
        F.date_format(F.from_utc_timestamp("hit.fetched_at", TZ), "yyyyMMdd").cast("int").alias("fetched_date_key"),
        F.coalesce(F.col("channel.channel_key"), F.lit(-1)).cast("bigint").alias("channel_key"),
        F.col("hit.search_order").cast("string").alias("search_order"),
        F.col("hit.result_rank").cast("int").alias("result_rank"),
        F.lit(1).cast("tinyint").alias("hit_count"),
    )
)

## Add metadata

In [ ]:
insert_date = datetime.now()
etl_by = "Databricks"
result_df = result_df.withColumn("etl_date", F.lit(insert_date)).withColumn("etl_by", F.lit(etl_by))

## Add checksums

In [ ]:
primary_key = ["fetched_at", "keyword_normalized", "video_id"]
result_df = add_checksums(result_df, primary_key)

## Create table

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {path} (
        fetched_at TIMESTAMP, keyword_normalized STRING, video_id STRING,
        keyword_key BIGINT, video_key BIGINT, fetched_date_key INT, channel_key BIGINT,
        search_order STRING, result_rank INT, hit_count TINYINT,
        etl_date TIMESTAMP, etl_by STRING, primary_checksum STRING, record_checksum STRING
    ) USING DELTA
    COMMENT 'Factless event (poc upsert method): 1 video x 1 keyword x 1 search run. hit_count = 1 (additive), result_rank non-additive.'
""")
result_df = result_df.select(*spark.table(path).columns)

# Write to Table

**full**: overwrite (poc `full_refresh`). **incremental**: upsert (poc `upsert`) = MERGE on `primary_checksum`; update only when `record_checksum` changed; insert new rows. Rows are never deleted.

In [ ]:
if load_type == "full":
    full_refresh(result_df, path)
else:
    result_df.createOrReplaceTempView("source")
    spark.sql(f"""
        MERGE INTO {path} AS t
        USING source AS s
        ON t.primary_checksum = s.primary_checksum
        WHEN MATCHED AND t.record_checksum <> s.record_checksum THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

## Check: one row per grain

In [ ]:
n, grain = spark.sql(f"SELECT COUNT(*), COUNT(DISTINCT primary_checksum) FROM {path}").first()
assert n == grain, f"{path}: {n - grain} duplicate rows on (fetched_at, keyword_normalized, video_id)"
last = spark.sql(f"DESCRIBE HISTORY {path} LIMIT 1").select("operation", "operationMetrics").first()
print(f"✅ {path}: {n} rows ·", last["operation"], dict(last["operationMetrics"] or {}))